# Faster-Is-Slower Demo

This notebook probes a faster-is-slower bottleneck proxy with repeated runs across multiple seeds. It uses a dedicated scenario asset with **30 agents**, a **0.75 m doorway**, and a **3 m corridor**.

Reference: Garcimartin A. et al., *Experimental Evidence of the "Faster Is Slower" Effect*, [10.1016/j.trpro.2014.09.085](https://doi.org/10.1016/j.trpro.2014.09.085).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from jupedsim_scenarios import load_scenario, run_sweep

## 1. Scenario Setup

We load a dedicated faster-is-slower proxy scenario from `scenario_files/faster-is-slower`, then vary only the desired speed $v_0$ through the scenario API.

In [ ]:
scenario = load_scenario("scenario_files/faster-is-slower")
scenario.set_agent_params(distribution_id=0, number=30, radius=0.15, desired_speed=1.2, distribution_mode="by_number")
scenario.set_seed(42)
scenario.set_max_time(1000)

#v0_values = np.array([0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4])
v0_values = np.array([0.8, 1.2, 1.6, 1.8])
seeds = [40, 41, 42, 43, 44]

print(scenario.summary())
print(f"Seeds: {seeds}")


## 2. Multi-Seed Sweep

Each desired-speed value is simulated across several seeds. The next cell stores all evacuation times so we can plot a mean curve with a 95% confidence interval.

In [ ]:
%%capture
# Sweep v0 × seed via jupedsim_scenarios.run_sweep — the library owns the
# cartesian product, per-trial scenario isolation, and result tabulation.
# (Previously this cell hand-rolled a nested loop with .copy()/.cleanup()
# bookkeeping; run_sweep handles all of that and runs trials in parallel.)
sweep = run_sweep(
    scenario,
    axes={"v0": v0_values.tolist()},
    apply={
        "v0": lambda s, v: s.set_agent_params(
            distribution_id=0, desired_speed=float(v)
        ),
    },
    seeds=seeds,
    workers=4,
)
df = sweep.to_dataframe()
sweep.cleanup()

## 3. Mean Evacuation Curve With Confidence Interval

The shaded band is a 95% confidence interval around the sample mean, computed from the seed-to-seed variation for each $v_0$.

In [ ]:
# Aggregate the sweep dataframe — one row per (v0, seed) trial.
agg = (
    df.groupby("v0")["evacuation_time"]
    .agg(["mean", "std", "count"])
    .reindex(v0_values)
)
means = agg["mean"].to_numpy()
stds = agg["std"].to_numpy()
cis = 1.96 * stds / np.sqrt(agg["count"].to_numpy())

plt.figure(figsize=(9, 5.5))
plt.plot(
    v0_values,
    means,
    marker="o",
    color="#C44E52",
    linewidth=2.5,
    markersize=8,
    label="Mean evacuation time",
)
plt.fill_between(
    v0_values,
    means - cis,
    means + cis,
    color="#C44E52",
    alpha=0.18,
    label="95% confidence interval",
)

for x, y in zip(v0_values, means, strict=False):
    plt.text(x, y + 1.6, f"{y:.1f}", ha="center", color="#8B2F34", fontsize=10)

plt.xlabel("Desired speed $v_0$ [m/s]", fontsize=12)
plt.ylabel("Evacuation time [s]", fontsize=12)
plt.title("Faster-Is-Slower: Mean Evacuation Time vs. Desired Speed", fontsize=14, pad=12)
plt.grid(axis="y", linestyle="--", alpha=0.4)
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

for v0 in v0_values:
    vals = df.loc[df["v0"] == v0, "evacuation_time"].to_numpy()
    print(f"v0={v0:.1f} m/s -> mean={vals.mean():.2f}s, std={vals.std(ddof=1):.2f}s, samples={np.round(vals, 2)}")